# GlassCut – Live Dataset Performance Profiling

This notebook measures **where time is actually spent** when converting a slide to tiles with `LiveSlideDataset`.

Two complementary profilers are used:
* **`pyinstrument`** – call-stack sampling (wall-clock, zero configuration)
* **`line_profiler`** – line-by-line timing of the hottest functions

Install once:
```bash
uv add --group dev pyinstrument line_profiler
```

In [1]:
# ── stdlib ──────────────────────────────────────────────────────────────
import time
import copy

# ── glasscut ────────────────────────────────────────────────────────────
from glasscut import Slide
from glasscut.data import breast_tissue
from glasscut.dataset import LiveSlideDataset
from glasscut.tiler import GridTiler
from glasscut.tissue_detectors import OtsuTissueDetector
from glasscut.slides.utils import magnification_to_level

# Grab the cached slide used in the usage guide
_, slide_path = breast_tissue()
print(f"Slide: {slide_path}")

/home/camilo/GlassCut/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Slide: /home/camilo/.cache/glasscut-data/tcga/breast/TCGA-A8-A082-01A-01-TS1.3cad4a77-47a6-4658-becf-d8cffa161d3a.svs


## 1. Baseline wall-clock time

In [2]:
tiler = GridTiler(
    tile_size=(512, 512),
    magnification=20,
    min_tissue_ratio=0.2,
    show_progress=True,
)

dataset = LiveSlideDataset(
    slide_paths=[str(slide_path)],
    tiler=tiler,
    stain_normalizer=None,  # leave stain normalisation OFF for now
    n_workers=4,
    batch_size=128,
    use_cucim=True,
)

t0 = time.perf_counter()
sample = dataset[0]
elapsed = time.perf_counter() - t0

print(f"\n✓ {len(sample.tiles)} tiles extracted in {elapsed:.2f}s")

Extracting tile batches:   0%|          | 0/7 [00:00<?, ?batch/s]

Extracting tile batches: 100%|██████████| 7/7 [00:04<00:00,  1.56batch/s]


✓ 883 tiles extracted in 4.79s


## 2. pyinstrument – call-stack profiler (no code changes needed)

Gives a flamegraph-style view of where wall time goes.

In [3]:
from pyinstrument import Profiler

tiler2 = GridTiler(
    tile_size=(512, 512), magnification=20, min_tissue_ratio=0.2, show_progress=False
)
dataset2 = LiveSlideDataset(
    slide_paths=[str(slide_path)],
    tiler=tiler2,
    stain_normalizer=None,
    n_workers=4,
    batch_size=128,
    use_cucim=True,
)

profiler = Profiler()
with profiler:
    _ = dataset2[0]

profiler.print(color=True)


  _     ._   __/__   _ _  _  _ _/_   Recorded: 18:54:40  Samples:  962
 /_//_/// /_\ / //_// / //_'/ //     Duration: 4.645     CPU time: 18.245
/   _/                      v5.1.2

Profile at /tmp/ipykernel_2896092/2021306825.py:16

4.645 <module>  /tmp/ipykernel_2896092/2021306825.py:1
└─ 4.644 LiveSlideDataset.__getitem__  ../../glasscut/dataset/live.py:85
   └─ 4.635 GridTiler.extract  ../../glasscut/tiler/grid.py:136
      ├─ 4.512 Slide.extract_tiles  ../../glasscut/slides/slide.py:256
      │  └─ 4.482 result_iterator  concurrent/futures/_base.py:612
      │        [4 frames hidden]  concurrent, threading
      │           4.412 lock.acquire  <built-in>
      └─ 0.121 GridTiler.get_tile_candidates  ../../glasscut/tiler/grid.py:68
         └─ 0.112 GridTiler._slide_tissue_mask  ../../glasscut/tiler/grid.py:188
            └─ 0.111 Slide.thumbnail  ../../glasscut/slides/slide.py:157
               └─ 0.111 OpenSlideBackend.get_thumbnail  ../../glasscut/slides/backends/openslide_ba

## 3. Manual phase breakdown

Break the pipeline into **3 phases** and time each independently:

| Phase | What happens |
|---|---|
| A – Thumbnail + tissue mask | `slide.thumbnail` + `OtsuTissueDetector.detect()` |
| B – Candidate selection | Loop over grid, sample mask, filter by ratio |
| C – Extraction + transforms | `GridTiler._extract_and_transform` via `ThreadPoolExecutor` (fused I/O + transforms) |


In [ ]:
import math
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from glasscut.tile import Tile

TILE_SIZE = (512, 512)
MAGNIFICATION = 20
MIN_TISSUE_RATIO = 0.2
N_WORKERS = 4
BATCH_SIZE = 128

detector = OtsuTissueDetector()
times: dict[str, float] = {}

tiler_ph = GridTiler(
    tile_size=TILE_SIZE,
    magnification=MAGNIFICATION,
    min_tissue_ratio=MIN_TISSUE_RATIO,
    show_progress=False,
)

with Slide(str(slide_path), use_cucim=True) as slide:

    # ── Phase A: thumbnail + tissue mask ──────────────────────────────────
    t = time.perf_counter()
    thumbnail = slide.thumbnail                          # lazyproperty – first access
    tissue_mask = np.asarray(detector.detect(thumbnail)) > 0
    times["A_thumbnail+mask"] = time.perf_counter() - t

    # ── Phase B: candidate selection ──────────────────────────────────────
    t = time.perf_counter()
    candidates = tiler_ph.get_tile_candidates(slide)
    times["B_candidate_selection"] = time.perf_counter() - t
    print(f"Candidates: {len(candidates)}")

    # ── Phase C: fused extraction + transforms (GridTiler owns the pool) ──
    # extract_tiles no longer lives on Slide; GridTiler._extract_and_transform
    # handles read_region + tissue ratio + transform pipeline in each thread.
    t = time.perf_counter()
    all_tiles: list[Tile] = []
    for start in range(0, len(candidates), BATCH_SIZE):
        batch = candidates[start : start + BATCH_SIZE]
        with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
            batch_tiles = list(
                executor.map(
                    lambda item: tiler_ph._extract_and_transform(slide, item),
                    batch,
                )
            )
        all_tiles.extend(batch_tiles)
    times["C_extraction+transforms"] = time.perf_counter() - t

total = sum(times.values())
print("\n── Phase breakdown ─────────────────────────────────────────")
for phase, t in times.items():
    print(f"  {phase:<30s}  {t:6.3f}s  ({100*t/total:.1f}%)")
print(f"  {'TOTAL':<30s}  {total:6.3f}s")

## 4. line_profiler – per-line timing of the hottest functions

In [5]:
%load_ext line_profiler

from glasscut.tiler.grid import GridTiler

tiler3 = GridTiler(
    tile_size=(512, 512), magnification=20, min_tissue_ratio=0.2, show_progress=False
)

with Slide(str(slide_path), use_cucim=True) as slide3:
    # Profile candidate selection (Phase B)
    %lprun -f tiler3.get_tile_candidates tiler3.get_tile_candidates(slide3)

Timer unit: 1e-09 s

Total time: 0.137086 s
File: /home/camilo/GlassCut/glasscut/tiler/grid.py
Function: GridTiler.get_tile_candidates at line 68

Line #      Hits         Time  Per Hit   % Time  Line Contents
    68                                               def get_tile_candidates(
    69                                                   self,
    70                                                   slide: Slide,
    71                                               ) -> list[tuple[int, int, int, int, float]]:
    72                                                   """Return preselected boxes with tissue ratio as ``(x, y, w, h, ratio)``."""
    73         1       7398.0   7398.0      0.0          self._validate_overlap(self.overlap, self.tile_size)
    74                                           
    75         1     633535.0 633535.0      0.5          level = magnification_to_level(self.magnification, slide.magnifications)
    76         1        260.0    260.0      0.0         

In [ ]:
from glasscut.tiler.grid import GridTiler

tiler4 = GridTiler(
    tile_size=(512, 512), magnification=20, min_tissue_ratio=0.2, show_progress=False
)

with Slide(str(slide_path), use_cucim=True) as slide4:
    candidates4 = tiler4.get_tile_candidates(slide4)

    # Profile the fused per-tile worker (Phase C inner loop):
    # _extract_and_transform does: extract_tile → set_tissue_ratio → transforms
    %lprun -f tiler4._extract_and_transform -f slide4.extract_tile list(map(lambda c: tiler4._extract_and_transform(slide4, c), candidates4[:BATCH_SIZE]))

## 5. Single extract_tile read-region timing

Isolates the raw I/O cost.

In [7]:
import statistics

with Slide(str(slide_path), use_cucim=True) as slide5:
    candidates5 = GridTiler(
        tile_size=(512, 512), magnification=20, min_tissue_ratio=0.2, show_progress=False
    ).get_tile_candidates(slide5)
    sample_coords = [(x, y) for x, y, _, _, _ in candidates5[:50]]

    times_single: list[float] = []
    for coords in sample_coords:
        t = time.perf_counter()
        slide5.extract_tile(coords, (512, 512), 20)
        times_single.append(time.perf_counter() - t)

print(f"Single tile (n=50):")
print(f"  mean  = {statistics.mean(times_single)*1000:.1f} ms")
print(f"  median= {statistics.median(times_single)*1000:.1f} ms")
print(f"  stdev = {statistics.stdev(times_single)*1000:.1f} ms")
print(f"  min   = {min(times_single)*1000:.1f} ms")
print(f"  max   = {max(times_single)*1000:.1f} ms")

Single tile (n=50):
  mean  = 19.1 ms
  median= 18.7 ms
  stdev = 1.0 ms
  min   = 18.4 ms
  max   = 23.1 ms


## 6. Worker count sweep – find the parallelism sweet spot

In [ ]:
from concurrent.futures import ThreadPoolExecutor

results: dict[int, float] = {}

tiler6 = GridTiler(
    tile_size=(512, 512), magnification=20, min_tissue_ratio=0.2, show_progress=False
)

with Slide(str(slide_path), use_cucim=True) as slide6:
    candidates6 = tiler6.get_tile_candidates(slide6)
    # Use only the first 256 candidates to keep the sweep fast
    sub = candidates6[:256]

    for n_w in [1, 2, 4, 8, 16, 32, 48, 64, 128]:
        t = time.perf_counter()
        with ThreadPoolExecutor(max_workers=n_w) as executor:
            list(executor.map(lambda c: tiler6._extract_and_transform(slide6, c), sub))
        results[n_w] = time.perf_counter() - t

print("Workers | Time (s) | Tiles/s")
print("-" * 35)
for w, t in results.items():
    print(f"  {w:>4}  |  {t:5.3f}   |  {256/t:6.0f}")